# Experimento de 140 combinaciones, parte 3: regresión

Resultados de las 28 combinaciones de regresión del índice continuo de aptitud (7 modelos × 4 optimizadores). Las métricas salen de los 5
pliegues externos por bloques espaciales. Los hiperparámetros se eligen con el RMSE interno, que castiga más los errores grandes, los que
cambiarían la clase de una planta.

Después de las métricas vienen las pruebas sobre los residuos que pide la guía. Los datos no tienen tiempo, así que las pruebas pensadas para
series (ACF, Ljung-Box, BDS) se aplican a los residuos ordenados por una curva de Hilbert sobre latitud y longitud. Plantas vecinas en la
curva son vecinas en el mapa, y la dependencia que miden esas pruebas pasa a ser autocorrelación espacial. Se complementa con el I de Moran
(Moran, 1950), la medida estándar de autocorrelación espacial.

## Flujo del experimento

![Flujo del experimento](fig_pipeline_experimento.png)

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats as st
from statsmodels.stats.diagnostic import het_white, het_breuschpagan, acorr_ljungbox
from statsmodels.tsa.stattools import acf, bds
from statsmodels.graphics.gofplots import qqplot
from src import analisis, balancing, config, cv, evaluate, models, stats

maestra, res = analisis.cargar()
reg = res[res.tarea == "reg"]
b = analisis.base()
y = b.y_reg
ORDEN = [m for m in models.MODELOS["reg"] if m in set(reg.modelo)]
orden_h = np.argsort(stats.indice_hilbert(b.tabla.latitude.values, b.tabla.longitude.values), kind="stable")
print(f"Combinaciones completas de regresión: {len(reg)} de 28 | modelos: {ORDEN}")

## 1. Las combinaciones

R² externo medio ± desviación entre pliegues. El R² compara contra predecir el promedio: 0 es no aprender nada y 1 es acertar todo.

In [ ]:
def tabla(metrica, dec=3):
    m = reg.pivot_table(index="modelo", columns="optimizador", values=metrica)
    d = reg.pivot_table(index="modelo", columns="optimizador", values=metrica + "_de")
    t = m.round(dec).astype(str) + " ± " + d.round(dec).astype(str)
    return t.reindex(ORDEN)[[c for c in ["grilla", "aleatoria", "bayesiana", "genetica"] if c in t.columns]]

display(tabla("r2"))
tabla("rmse", 4)

## 2. El mejor caso de cada modelo

Combinación elegida por puntaje interno (RMSE). RMSE y MAE están en las unidades del índice, que va de 0 a 1; las clases se cortan en 0.4 y 0.6.

In [ ]:
mejores = analisis.mejores_por_modelo(res, "reg").reindex(ORDEN)
t = mejores[["optimizador"]].copy()
for c in ["r2", "rmse", "mae"]:
    t[c] = mejores[c].round(4).astype(str) + " ± " + mejores[c + "_de"].round(4).astype(str)
t["RMSE interno"] = (-mejores.puntaje_interno).round(4)
t

## 3. Entrenamiento, validación y prueba

Para el mejor modelo, en el primer pliegue externo: los hiperparámetros elegidos se reentrenan y se grafican las predicciones en tres conjuntos.
- Entrenamiento: los datos con los que aprendió. Si aquí acierta mucho más que en prueba, hay sobreajuste.
- Validación: el primer pliegue interno (entrena con los otros dos).
- Prueba: el pliegue externo, bloques que nunca vio.

In [ ]:
mejor = mejores.puntaje_interno.idxmax()
fila = maestra[(maestra.tarea == "reg") & (maestra.modelo == mejor) & (maestra.optimizador == mejores.loc[mejor, "optimizador"]) & (maestra.pliegue == 0)].iloc[0]
params = json.loads(fila.params)
print("Mejor modelo:", mejor, "|", mejores.loc[mejor, "optimizador"], "|", params)

tr, te = cv.externas(b.y_clf, b.grupos)[0]
itr, iva = cv.internas(b.y_clf[tr], b.grupos[tr])[0]
def ajusta(idx):
    p = balancing.construir(models.estimador("reg", mejor), mejor, "ninguno").set_params(**{f"m__{k}": v for k, v in params.items()})
    return p.fit(b.X[idx], y[idx])
m_ext, m_int = ajusta(tr), ajusta(tr[itr])
conjuntos = {"entrenamiento": (y[tr], m_ext.predict(b.X[tr])), "validación": (y[tr][iva], m_int.predict(b.X[tr][iva])),
             "prueba": (y[te], m_ext.predict(b.X[te]))}
fig, axes = plt.subplots(1, 3, figsize=(18, 5.5), sharex=True, sharey=True)
for ax, (nombre, (real, pred)) in zip(axes, conjuntos.items()):
    met = evaluate.metricas_reg(real, pred)
    ax.hexbin(real, pred, gridsize=60, bins="log", cmap="Reds", mincnt=1)
    ax.plot([0, 1], [0, 1], "k--", lw=.8)
    for c in (0.4, 0.6):
        ax.axvline(c, color="grey", lw=.5); ax.axhline(c, color="grey", lw=.5)
    ax.set_title(f"{nombre}: R² {met['r2']:.3f}, RMSE {met['rmse']:.4f}"); ax.set_xlabel("índice real")
axes[0].set_ylabel("índice predicho")
fig.suptitle(f"{mejor}: predicho contra real (pliegue externo 1)", y=1.02)
fig.tight_layout(); fig.savefig("fig_exp_reg_tvp.png", dpi=130, bbox_inches="tight"); plt.show()

## 4. Residuos del mejor modelo

Residuo = real − predicho, con las predicciones fuera de pliegue (cada planta predicha por el modelo que no la vio).

In [ ]:
pred = analisis.oof(mejores.loc[mejor])["pred"]
e = y - pred
eh = e[orden_h]
fig, axes = plt.subplots(1, 3, figsize=(18, 4.8))
axes[0].hist(e, bins=80, color="#C0141F", alpha=.8); axes[0].set_title("Histograma de los residuos")
qqplot(e, line="s", ax=axes[1], markersize=2, alpha=.3); axes[1].set_title("Q-Q contra la normal")
lags = 40
axes[2].bar(range(1, lags + 1), acf(eh, nlags=lags)[1:], color="#1A1A1A")
axes[2].axhline(1.96 / np.sqrt(len(eh)), color="r", ls="--", lw=.8); axes[2].axhline(-1.96 / np.sqrt(len(eh)), color="r", ls="--", lw=.8)
axes[2].set_title("ACF de los residuos en orden de Hilbert"); axes[2].set_xlabel("rezago (vecinos en la curva)")
fig.tight_layout(); fig.savefig("fig_exp_reg_residuos.png", dpi=130, bbox_inches="tight"); plt.show()

In [ ]:
Xc = np.column_stack([np.ones(len(e)), (b.tabla[config.VARIABLES].fillna(b.tabla[config.VARIABLES].median()).values - 0)])
Xc[:, 1:] = (Xc[:, 1:] - Xc[:, 1:].mean(0)) / Xc[:, 1:].std(0)
lb = acorr_ljungbox(eh, lags=[1, 5, 20], return_df=True)
jb = st.jarque_bera(e)
bp = het_breuschpagan(e, Xc)
wh = het_white(e, Xc[:, :8])  # White con las 7 primeras variables: con las 15 serían 135 regresores
ventanas = [eh[i:i + 2500] for i in np.linspace(0, len(eh) - 2500, 4).astype(int)]
bds_p = [float(np.atleast_1d(bds(v, max_dim=2)[1])[0]) for v in ventanas]
moran = stats.moran_i(e, b.tabla.latitude.values, b.tabla.longitude.values, permutaciones=199)
pruebas = pd.DataFrame([
    {"prueba": "Jarque-Bera", "hipótesis nula": "residuos normales", "estadístico": jb.statistic, "p": jb.pvalue},
    {"prueba": "Breusch-Pagan", "hipótesis nula": "varianza constante", "estadístico": bp[0], "p": bp[1]},
    {"prueba": "White (7 variables)", "hipótesis nula": "varianza constante", "estadístico": wh[0], "p": wh[1]},
    *[{"prueba": f"Ljung-Box, rezago {l}", "hipótesis nula": "sin autocorrelación en orden de Hilbert", "estadístico": r.lb_stat, "p": r.lb_pvalue} for l, r in lb.iterrows()],
    *[{"prueba": f"BDS, ventana {i + 1} de 2,500", "hipótesis nula": "residuos i.i.d. en orden de Hilbert", "estadístico": np.nan, "p": p} for i, p in enumerate(bds_p)],
    {"prueba": "I de Moran (8 vecinos)", "hipótesis nula": "sin autocorrelación espacial", "estadístico": moran["I"], "p": moran["p"]},
]).set_index("prueba")
pruebas

Con casi 58 mil observaciones, cualquier desviación pequeña sale significativa. Por eso conviene leer también el tamaño: el I de Moran va de −1 a 1 y
la ACF muestra cuánto se parecen los residuos de vecinos. Las mismas pruebas para el mejor caso de cada modelo:

In [ ]:
filas = []
for m in ORDEN:
    em = y - analisis.oof(mejores.loc[m])["pred"]
    mo = stats.moran_i(em, b.tabla.latitude.values, b.tabla.longitude.values, permutaciones=99)
    filas.append({"modelo": m, "R² externo": mejores.loc[m, "r2"], "ACF rezago 1 (Hilbert)": acf(em[orden_h], nlags=1)[1],
                  "I de Moran": mo["I"], "p Moran": mo["p"], "sesgo medio": em.mean(), "asimetría": st.skew(em), "curtosis": st.kurtosis(em)})
pd.DataFrame(filas).set_index("modelo").round(4)

## 5. Interpretación

Random Forest es el mejor (R² 0.902, RMSE 0.0416), seguido de XGBoost (0.899) y el árbol de decisión (0.880). KNN y el SVR RBF quedan cerca de 0.79, y Ridge y Lasso en 0.479 con cualquier optimizador, porque casi no tienen hiperparámetros que importen. En Random Forest el optimizador tampoco cambia nada (0.902 a 0.903). En XGBoost sí: va de 0.890 con grilla o genética a 0.899 con la bayesiana, y en el SVR RBF de 0.776 a 0.792.

Los residuos de Random Forest no son normales (asimetría de -1.03 y curtosis de 8.4, con una cola larga de errores por sobreestimar el índice) y no tienen varianza constante. Además conservan dependencia espacial: el I de Moran es 0.103 (p = 0.005) y la ACF en el rezago 1 es 0.162. En Ridge esos valores son 0.317 y 0.373. Con casi 58 mil plantas todas las pruebas rechazan, así que lo que informa es el tamaño: la dependencia de los residuos es pequeña en los árboles y los bosques (I de Moran de 0.10 a 0.12) y mucho mayor en los modelos lineales.

## Referencias

- Breusch, T. S. y Pagan, A. R. (1979). A simple test for heteroscedasticity and random coefficient variation. *Econometrica, 47*(5), 1287-1294. https://doi.org/10.2307/1911963
- Broock, W. A., Scheinkman, J. A., Dechert, W. D. y LeBaron, B. (1996). A test for independence based on the correlation dimension. *Econometric Reviews, 15*(3), 197-235. https://doi.org/10.1080/07474939608800353
- Jarque, C. M. y Bera, A. K. (1987). A test for normality of observations and regression residuals. *International Statistical Review, 55*(2), 163-172. https://doi.org/10.2307/1403192
- Ljung, G. M. y Box, G. E. P. (1978). On a measure of lack of fit in time series models. *Biometrika, 65*(2), 297-303. https://doi.org/10.1093/biomet/65.2.297
- Moran, P. A. P. (1950). Notes on continuous stochastic phenomena. *Biometrika, 37*(1/2), 17-23. https://doi.org/10.2307/2332142
- White, H. (1980). A heteroskedasticity-consistent covariance matrix estimator and a direct test for heteroskedasticity. *Econometrica, 48*(4), 817-838. https://doi.org/10.2307/1912934